# Notebook 06 — Walk-Forward Adaptive Equilibrium

## Objective

Extend the static fair-value framework developed in Notebook 05
using walk-forward retraining in order to test whether apparent
balancing-market disequilibrium persists once equilibrium is
allowed to evolve dynamically through time.

The notebook investigates whether:

- residual spread regimes reflect genuine persistent mispricing,
or instead:
- instability in the assumed static equilibrium relationship.

The framework repeatedly retrains the fair-value model using
only prior information, allowing equilibrium estimation to
adapt continuously to changing market conditions.

Primary outputs:

- walk-forward equilibrium estimates,
- adaptive residual diagnostics,
and comparison with the earlier static framework.

## Motivation

Notebook 05 identified several apparent residual stress regimes
using a static XGBoost fair-value model.

However, static equilibrium assumptions may be inappropriate
in balancing markets where:
- operational conditions evolve continuously,
- volatility regimes shift,
- and system stress changes through time.

This notebook therefore tests whether apparent persistent
disequilibrium survives once equilibrium estimation itself
is allowed to adapt dynamically.

The central research question becomes:

```text
Are balancing spreads persistently mispriced,
or is equilibrium simply nonstationary?
```

## 1 — Load Dataset

### Static Fair-Value Dependency

This notebook compares the adaptive walk-forward framework
against the static equilibrium model developed in Notebook 05.

The file:

```python
data/processed/static_fair_value_results.parquet
```

is generated at the end of Notebook 05 and contains:
- static fair-value predictions,
- timestamps,
- and residual diagnostics.

Notebook 05 must therefore be executed before running the
comparison sections of this notebook.

In [1]:
import pandas as pd
import numpy as np

market = pd.read_parquet(

    "../data/processed/continuous_market_state.parquet"
)

results = pd.read_parquet(
    "../data/processed/static_fair_value_results.parquet"
)

market['timestamp'] = pd.to_datetime(
    market['timestamp']
)

print(market.shape)

(17525, 33)


## 2 — Define Features + Target

In [2]:
target = 'basis_spread_clipped'

features = [

    'wind_mw',
    'sw_demand_mw',

    'total_action_mw',
    'action_count',

    'spread_lag_1',
    'spread_lag_48',

    'spread_rolling_mean_4',
    'spread_volatility_4',
    'spread_volatility_12',

    'wind_ramp_1',
    'wind_ramp_4',

    'demand_ramp_1',
    'demand_ramp_4'
]

In [3]:
model_df = market[
    ['timestamp'] + features + [target]
].dropna()

print(model_df.shape)

(17415, 15)


### Methodological Caveat

Several explanatory variables partially encode contemporaneous
balancing conditions.

As discussed in Notebook 05, strong explanatory performance
does not necessarily imply deployable predictive power.

The objective of this notebook is therefore not:
- high-frequency forecasting,
but rather:
- investigation of equilibrium stability through time.

The walk-forward framework uses an expanding historical
training window rather than a fixed rolling window.

This allows the model to:
- retain longer-term operational information,
- stabilise equilibrium estimation,
- and gradually adapt to evolving market conditions.

A fixed rolling window would adapt more aggressively,
but may overreact to temporary market regimes.

## 3. Walk-Forward Adaptive Equilibrium Framework

The model is retrained repeatedly through time using only
historically available information.

Each iteration uses:
- an expanding historical training window
- followed by a forward test window.

This prevents:
- look-ahead bias,
- static equilibrium assumptions,
- and artificial persistence generated by fixed training periods.

The framework therefore approximates a continuously adapting
market participant recalibrating equilibrium expectations
through time.

In [4]:
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error
)

In [5]:
walk_preds = []

walk_actuals = []

walk_timestamps = []

In [6]:
dates = model_df['timestamp'].sort_values()

start_date = dates.min()

end_date = dates.max()

### Window Structure

Each walk-forward iteration uses:

| Component | Length |
|---|---|
| Training window | 90 days |
| Forward test window | 7 days |
| Retraining frequency | Weekly |

This structure allows equilibrium estimation to evolve gradually
while preserving sufficient historical context for stable training.

In [7]:
current_start = start_date

while current_start < end_date:

    train_end = current_start + pd.Timedelta(days=90)

    test_end = train_end + pd.Timedelta(days=7)

    train_df = model_df[
        model_df['timestamp'] < train_end
    ]

    test_df = model_df[
        (model_df['timestamp'] >= train_end)
        &
        (model_df['timestamp'] < test_end)
    ]

    # Skip empty windows
    if len(test_df) == 0:
        break

    # =====================================================
    # TRAIN MODEL
    # =====================================================

    xgb = XGBRegressor(

        n_estimators=300,

        max_depth=4,

        learning_rate=0.05,

        subsample=0.8,

        colsample_bytree=0.8,

        random_state=42
    )

    xgb.fit(

        train_df[features],

        train_df[target]
    )

    # =====================================================
    # PREDICT
    # =====================================================

    preds = xgb.predict(
        test_df[features]
    )

    walk_preds.extend(preds)

    walk_actuals.extend(
        test_df[target]
    )

    walk_timestamps.extend(
        test_df['timestamp']
    )

    # =====================================================
    # MOVE WINDOW FORWARD
    # =====================================================

    current_start += pd.Timedelta(days=7)

## 4 — Build walk-forward results

In [8]:
walk_results = pd.DataFrame({

    'timestamp': walk_timestamps,

    'actual_spread': walk_actuals,

    'predicted_fair_value': walk_preds
})

walk_results['spread_residual'] = (

    walk_results['actual_spread']

    - walk_results['predicted_fair_value']
)

walk_results['abs_residual'] = (

    walk_results['spread_residual']
    .abs()
)

walk_results = walk_results.sort_values(
    'timestamp'
)

In [9]:
walk_results = walk_results.rename(columns={'abs_residual': 'residual_abs'})


In [10]:
walk_compare = walk_results[
    walk_results['timestamp'] >= common_start
]


NameError: name 'common_start' is not defined

In [ ]:
mae = mean_absolute_error(

    walk_results['actual_spread'],

    walk_results['predicted_fair_value']
)

print(f"Walk-forward MAE: {mae:.2f}")

Direct comparison between static and walk-forward MAE should
be interpreted cautiously.

The static framework evaluates a single terminal holdout period,
whereas the walk-forward framework evaluates multiple sequential
forward windows across evolving market conditions.

The primary objective of the walk-forward framework is therefore:
- evaluation of equilibrium stability through time,
rather than:
- minimisation of average prediction error alone.

### Walk-Forward Performance Interpretation

Although the walk-forward framework produced a higher average
MAE than the earlier static framework, it provides a more
realistic chronological evaluation of equilibrium behaviour
through time.

The adaptive framework reduced some instability associated
with static equilibrium assumptions, but substantial residual
stress structure remained during the late-2022 period.

This suggests that:
- part of the apparent disequilibrium identified in Notebook 05
  reflected evolving equilibrium conditions,
while:
- part reflected genuinely persistent operational stress that
  remained difficult to model adaptively.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.figure(figsize=(14,6))

plt.plot(

    walk_results['timestamp'],

    walk_results['spread_residual'],

    alpha=0.7
)

plt.axhline(
    0,
    linestyle='--'
)

plt.gca().xaxis.set_major_locator(
    mdates.MonthLocator()
)

plt.gca().xaxis.set_major_formatter(
    mdates.DateFormatter('%Y-%m')
)

plt.xticks(rotation=45)

plt.title(
    "Walk-Forward Residuals Through Time"
)

plt.xlabel("Date")

plt.ylabel("Residual Spread")

plt.tight_layout()

plt.show()

## 5 — Static vs Adaptive Residual Comparison

The central question of the project is whether apparent
balancing-market disequilibrium persists once equilibrium
estimation itself is allowed to adapt dynamically.

This section directly compares:
- residual structure from the static fair-value framework,
with:
- residual structure from the adaptive walk-forward framework.

If balancing-market disequilibrium is genuinely persistent,
both models should exhibit similar residual clustering.

If instead equilibrium itself evolves through time,
the adaptive framework should materially reduce residual
instability during changing market conditions.

In [ ]:
common_start = results['timestamp'].min()

walk_compare = walk_results[
    walk_results['timestamp'] >= common_start
]

In [ ]:
static_smooth = (

    results
    .set_index('timestamp')['residual_abs']
    .rolling(48)
    .mean()
)

walk_smooth = (

    walk_compare
    .set_index('timestamp')['residual_abs']
    .rolling(48)
    .mean()
)

In [ ]:
plt.figure(figsize=(14,6))

plt.plot(

    static_smooth.index,
    static_smooth,

    label='Static Fair Value',
    linewidth=2
)

plt.plot(

    walk_smooth.index,
    walk_smooth,

    label='Walk-Forward Adaptive',
    linewidth=2
)

plt.legend()

plt.title(
    "Static vs Adaptive Residual Volatility"
)

plt.xlabel("Date")

plt.ylabel("48-Period Rolling Absolute Residual")

plt.show()

In [ ]:
print(results['timestamp'].min())
print(results['timestamp'].max())

In [ ]:
print(walk_compare['timestamp'].min())
print(walk_compare['timestamp'].max())

### Interpretation

Both the static and adaptive equilibrium frameworks exhibit
similar residual stress expansion during the late-2022 period.

This suggests that:
- a meaningful portion of residual structure reflects genuine
  operational stress conditions,
rather than:
- purely static-equilibrium misspecification.

The adaptive framework partially reduced residual instability
during calmer operational periods, but substantial residual
stress persisted during the late-2022 stress regime.

This suggests that adaptive equilibrium estimation explained
part of the apparent disequilibrium identified by the static
framework, but did not fully eliminate residual stress structure.

This indicates that balancing-market stress during late 2022
was only partially captured by evolving equilibrium estimation.

### Broader Market Context

The evaluation period includes the late-2022 European energy
crisis, characterised by:
- elevated gas-market volatility,
- stressed generation availability,
- and unusually unstable power-market conditions.

Residual stress persistence during this period may therefore
partly reflect broader macro-energy instability rather than
Balancing Mechanism dynamics alone.

## 6 — Conclusions

The walk-forward adaptive equilibrium framework provided a
more realistic evaluation of balancing-market dynamics under
evolving operational conditions.

Although the adaptive framework modestly stabilised residual
behaviour through time, substantial residual stress structure
persisted during the late-2022 stress regime.

This suggests that:
- some apparent disequilibrium identified by the static model
  reflected evolving equilibrium conditions,
while:
- a meaningful component of residual structure reflected
  genuine operational stress that remained difficult to model.

### 6.1 Market Interpretation

The GB Balancing Mechanism appears neither perfectly efficient
nor persistently mispriced.

Balancing spreads exhibit:
- persistence,
- volatility clustering,
- and operational state dependence,

with residual stress becoming most visible during extreme
market conditions.

The similarity between static and adaptive residual structure
during late 2022 suggests that severe balancing stress was not
purely an artefact of static-equilibrium misspecification.

### 6.2 Methodological Interpretation

The notebook demonstrates the importance of testing equilibrium
stability explicitly within nonstationary energy markets.

Walk-forward retraining introduced:
- temporal discipline,
- adaptive equilibrium estimation,
- and more realistic forward evaluation.

However, adaptive retraining alone did not fully eliminate
residual stress clustering, suggesting that some balancing-market
stress regimes reflect genuinely difficult operational conditions
rather than purely model drift.

### 6.3 Final Interpretation

The evidence appears most supportive of a hybrid interpretation:

- balancing-market equilibrium evolves through time,
- adaptive modelling partially improves equilibrium estimation,
but:
- substantial residual stress structure persists during periods
  of extreme operational and macro-energy stress.

The GB Balancing Mechanism therefore appears characterised by:
- partial equilibrium adaptation,
- rapid operational repricing,
- and intermittent stress regimes that remain difficult to fully
  capture using market-state variables alone.